# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore and process the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. All entities such as record sets, fields, and columns are referenced by their `@id` for full reproducibility and metadata traceability.

### Dataset Source
The Croissant schema for this dataset is published at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Install mlcroissant if necessary (uncomment in Colab or where needed)
!pip install mlcroissant

## 1. Data Loading

Load the dataset metadata with `mlcroissant`, and view general information such as its title and description.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import warnings
warnings.filterwarnings('ignore')  # for cleaner output in notebook

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access metadata
metadata = dataset.metadata  # This is a dataclass, not a dict
print(f"Dataset: {metadata.name}\nDescription: {metadata.description}\nLicense: {metadata.license}\n")


## 2. Data Overview

Review record set, field, and column `@id`s present in the dataset. This helps us understand the main data tables and their structure.

> ⚠️ Note: For Croissant, `record sets` are the main data tables. They reference fields which map to columns in files. All IDs are referenced by their `@id` attributes.

In [ ]:
# List all Record Sets, their Fields, and Field details by their `@id`
record_sets = metadata.record_sets
print(f"Total record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"RecordSet @id: {rs.id}")
    print(f"  Name: {rs.name}")
    print(f"  Description: {rs.description}")
    print(f"  Fields:")
    for field in rs.fields:
        print(f"    Field @id: {field.id}")
        print(f"      Name: {field.name}")
        print(f"      Data type: {field.data_type}")
        print(f"      Source column @id: {field.column}")
    print("-")

## 3. Data Extraction

For each record set, load all records (i.e., rows) into a pandas DataFrame. Columns/fields are referenced by their `@id`. This enables flexible and reproducible downstream analysis.

Use the printed overview to select a record set and its field(s) of interest.

In [ ]:
# Prepare DataFrames for each Record Set by `@id`
dataframes = {}

# Collect all record set IDs
record_set_ids = [rs.id for rs in metadata.record_sets]

for rs_id in record_set_ids:
    # Records are dictionaries keyed by field IDs (e.g. '@id' of each field)
    records = list(dataset.records(record_set=rs_id))
    if records:
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records for RecordSet {rs_id} with columns:")
        print(dataframes[rs_id].columns.tolist())
    else:
        print(f"No records found for RecordSet {rs_id}")
    print('-'*40)

# For demonstration, use the first available record set
if len(record_set_ids) > 0 and record_set_ids[0] in dataframes:
    example_rs_id = record_set_ids[0]
    print(f"First 5 rows for RecordSet {example_rs_id}:")
    display(dataframes[example_rs_id].head())

## 4. Exploratory Data Analysis (EDA)

We'll demonstrate numeric filtering and grouping using fields referenced by their `@id`. Adjust these to target the correct numeric or categorical fields (see the overview above for possible values).

*Example*: Filter for proteins with high coverage and analyze distributions by group.

In [ ]:
# Identify valid numeric and group fields from the fields overview (by @id)
# Example (to be adjusted to the actual field IDs:
# Suppose we found a field with @id = 'coverage', and want to group by 'sample_type'

# (Replace these with actual @id values from section 2)
record_set_id = example_rs_id  # Chosen in previous section
df = dataframes[record_set_id]

# Try common alternatives for demonstration (user should use the @id actually present in their schema)
possible_numeric_ids = [col for col in df.columns if 'coverage' in col.lower() or 'abundance' in col.lower() or 'mw' in col.lower() or 'count' in col.lower() or 'peptide' in col.lower()]
numeric_field_id = possible_numeric_ids[0] if possible_numeric_ids else df.columns[0]  # fallback

print(f"Analyzing numeric field (by @id): {numeric_field_id}")

# Filter: numeric_field_id > threshold
threshold = df[numeric_field_id].astype(float).mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0
filtered_df = df[df[numeric_field_id].astype(float) > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
display(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id].astype(float) - filtered_df[numeric_field_id].astype(float).mean()
) / filtered_df[numeric_field_id].astype(float).std()
print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try to group by another (likely categorical) field
possible_group_ids = [col for col in df.columns if col != numeric_field_id and (('sample' in col.lower()) or ('type' in col.lower()) or ('category' in col.lower()))]
group_field_id = possible_group_ids[0] if possible_group_ids else None

if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
    print(f"Grouped data by {group_field_id} (first five rows):")
    display(grouped_df.head())
else:
    print("No suitable categorical group field was found by heuristic.")

## 5. Visualization

Let's visualize the distribution of a numeric field (referenced by its `@id`) and, if grouping exists, compare means by category. For more advanced visualizations, cross-reference available field IDs.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of selected numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field_id].astype(float), kde=True, bins=20)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.tight_layout()
plt.show()

# If group_field_id exists, draw boxplot
if group_field_id and group_field_id in df.columns:
    plt.figure(figsize=(10, 5))
    sns.boxplot(x=df[group_field_id], y=df[numeric_field_id].astype(float))
    plt.xticks(rotation=45, ha='right')
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.tight_layout()
    plt.show()

## 6. Conclusion

In this notebook, we demonstrated how to load, explore, and analyze a FAIR² dataset using the `mlcroissant` library. We referenced all schema elements by their `@id` as recommended, performed filtering and normalization of numeric fields, and visualized their distributions. Adjust field and group selections to deeper explore biological phenomena in the dataset.
